# DataRetail360: flujo completo Bronze, Silver y Gold

Ejecutamos las mismas funciones del proceso completo, una etapa a la vez.
Bronze conserva los CSV, Silver genera ocho entidades con claves de negocio
y Gold construye tres hechos, incluido el snapshot de stock, después de releer Parquet.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

RAIZ = Path.cwd().resolve()
if not (RAIZ / "src" / "proceso.py").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from src.proceso import SesionETL
from src.carga.gold.conexion import crear_motor

sesion = SesionETL(RAIZ, modo="manual")
print("Ejecución:", sesion.id_ejecucion)

## 1. Extraer Bronze

Verificamos columnas, filas y SHA-256 de los ocho CSV. Todavía no se
normaliza ni se descarta ninguna fila.

In [ ]:
fuentes = sesion.extraer()
display(pd.DataFrame([
    {"fuente": nombre, "filas": len(datos), "columnas": len(datos.columns)}
    for nombre, datos in fuentes.items()
]))
display(fuentes["ventas"].head(3))

## 2. Limpiar y conciliar Silver

Normalizamos identificadores y tipos, retiramos copias exactas y enviamos
errores a auditoría. Ventas, devoluciones y stock conservan claves de negocio;
aún no existen claves sustitutas ni tablas dimensionales.

In [ ]:
tablas_silver = sesion.limpiar_silver()
auditoria = sesion.conciliar_silver()
display(pd.DataFrame([
    {"entidad_silver": nombre, "filas": len(datos)}
    for nombre, datos in tablas_silver.items()
]))
display(pd.DataFrame(auditoria["conciliacion"]))
display(pd.DataFrame(auditoria["rechazos"])[["fuente", "motivo"]].value_counts().reset_index(name="filas"))

## 3. Guardar y volver a leer Silver

Escribimos ocho Parquet Snappy, auditoría y manifiesto. Después descartamos
la copia previa y comprobamos que Gold usa la lectura física.

In [ ]:
manifiesto = sesion.guardar_silver()
display(pd.DataFrame(manifiesto["entidades"])[["entidad", "archivo", "filas", "sha256"]])

tablas_silver = sesion.leer_silver()
claves_gold = {"clave_fecha", "clave_producto", "clave_cliente",
               "clave_sucursal_canal", "clave_venta", "clave_devolucion"}
assert all(not claves_gold.intersection(datos.columns) for datos in tablas_silver.values())
print("Silver releída desde disco y sin claves sustitutas Gold.")

## 4. Construir Gold

Aplicamos SCD 1 y 3 a productos, SCD 2 a clientes, construimos calendario
y resolvemos las claves de los dos hechos.

In [ ]:
dimensiones = sesion.construir_dimensiones_gold()
hechos = sesion.construir_hechos_gold()
tablas_gold = sesion.conciliar_gold()
display(pd.DataFrame([
    {"tabla_gold": nombre, "filas": len(datos)}
    for nombre, datos in tablas_gold.items()
]))
display(dimensiones["dim_cliente"].head(3))
display(hechos["hecho_ventas"].head(3))
display(hechos["hecho_stock_diario"].head(3))

## 5. Cargar, validar y publicar SQLite

Creamos una base temporal, cargamos dimensiones antes que hechos, ejecutamos
controles SQL y reemplazamos la base final únicamente si todos terminan en cero.

In [ ]:
sesion.conectar()
sesion.crear_modelo()
sesion.cargar_dimensiones()
sesion.cargar_hechos()
controles = sesion.validar()
display(pd.DataFrame([
    {"control": nombre, "infracciones": valor}
    for nombre, valor in controles.items()
]))
display(pd.Series(sesion.resumen["metricas"], name="valor").to_frame())

In [ ]:
resumen = sesion.publicar()
assert resumen["estado"] == "CORRECTA"
print("Base publicada:", sesion.ruta_base)
print("Ventas Silver/Gold:", resumen["conciliacion_gold"]["ventas_silver"],
      resumen["conciliacion_gold"]["ventas_gold"])
print("Devoluciones Silver/Gold:", resumen["conciliacion_gold"]["devoluciones_silver"],
      resumen["conciliacion_gold"]["devoluciones_gold"])
print("Stock Silver/Gold:", resumen["conciliacion_gold"]["stock_silver"],
      resumen["conciliacion_gold"]["stock_gold"])

## 6. Consultar el resultado

Las vistas evitan multiplicar ventas cuando existen varias devoluciones y
permiten explicar actividad comercial y stock por producto y tienda.

In [ ]:
motor = crear_motor(sesion.ruta_base)
try:
    with motor.connect() as conexion:
        display(pd.read_sql_query(
            "SELECT anio, mes, canal, ventas_netas "
            "FROM vista_actividad_mensual ORDER BY anio, mes, canal LIMIT 12",
            conexion,
        ))
        display(pd.read_sql_query(
            "SELECT codigo_sucursal, codigo_producto, stock_disponible, estado_stock "
            "FROM vista_stock_tiendas ORDER BY stock_disponible LIMIT 10",
            conexion,
        ))
        display(pd.read_sql_query(
            "SELECT codigo_producto, nombre_producto, ventas_netas "
            "FROM vista_ventas_por_producto ORDER BY ventas_netas DESC LIMIT 10",
            conexion,
        ))
finally:
    motor.dispose()